# Reconstrucción de frontend desde capturas

El laboratorio descarga pares de captura/HTML de WebSight, entrena un índice visual k-NN y guarda el artefacto local. La API recupera ejemplos visualmente cercanos y los usa como contexto para un modelo de visión que genera HTML, CSS y JavaScript. El índice es un modelo de recuperación; el notebook no ajusta los pesos de un modelo generativo.

## 1. Entorno Windows con Anaconda

En Anaconda Prompt (CMD), desde la carpeta del proyecto: `conda create -n screenshot-ui python=3.11 -y`, `conda activate screenshot-ui`, `python -m pip install -r requirements.txt` y `python -m playwright install chromium`. Crea `.env` desde `.env.example` y guarda allí `OPENAI_API_KEY`; no muestres la clave en las celdas ni en sus salidas.

In [1]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / 'backend' / 'api.py').is_file() else HERE / '03_asistente_de_revision_de_codigo'
if not (ROOT / 'backend' / 'api.py').is_file():
    raise RuntimeError('Abre este notebook desde la carpeta del proyecto o su carpeta padre.')
sys.path.insert(0, str(ROOT))
print('Proyecto:', ROOT)

Proyecto: C:\Users\byron\Documents\Bases de datos 2\Proyecto_IA_asistente_de_revision_de_codigo\03_asistente_de_revision_de_codigo


## 2. Descargar una muestra de WebSight

WebSight v0.2 ofrece pares sintéticos de capturas y HTML con Tailwind. El dataset completo ocupa cientos de GB; este proyecto lee en streaming y guarda una muestra pequeña. Aun así, el primer shard remoto puede transferir cientos de MB. La licencia es CC BY 4.0; conserva la atribución si distribuyes el modelo o la aplicación.

In [2]:
from descargar_dataset import download_dataset

MAX_SAMPLES = 128
download_result = download_dataset(max_samples=MAX_SAMPLES, output_dir=ROOT / 'data' / 'processed')
download_result

C:\Users\byron\anaconda3\envs\Asistente-revision-de-codigo\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\byron\anaconda3\envs\Asistente-revision-de-codigo\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\byron\.cache\huggingface\hub\datasets--HuggingFaceM4--WebSight. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Pyth

{'samples': 128,
 'manifest': WindowsPath('C:/Users/byron/Documents/Bases de datos 2/Proyecto_IA_asistente_de_revision_de_codigo/03_asistente_de_revision_de_codigo/data/processed/examples.jsonl'),
 'dataset': 'HuggingFaceM4/WebSight'}

## 3. Entrenar el índice visual

Cada captura se representa como una miniatura RGB de 32×32 píxeles. `NearestNeighbors` aprende un índice coseno sobre la muestra; la API recupera los pares más cercanos y se los pasa como referencias al generador de visión. Este baseline es liviano para CPU y sirve para el laboratorio, pero no equivale a entrenar un VLM.

In [3]:
from backend.ml import train

model = train(ROOT / 'data' / 'processed', ROOT / 'models' / 'modelo.joblib')
print(f'Índice entrenado con {len(model["examples"])} pares.')
print('Artefacto:', ROOT / 'models' / 'modelo.joblib')

Índice entrenado con 128 pares.
Artefacto: C:\Users\byron\Documents\Bases de datos 2\Proyecto_IA_asistente_de_revision_de_codigo\03_asistente_de_revision_de_codigo\models\modelo.joblib


## 4. Comprobar recuperación con una imagen del dataset

Esta comprobación usa una muestra incluida en el conjunto para confirmar que el artefacto puede cargar la imagen y devolver su HTML asociado. No mide generalización; para evaluar rendimiento, separa un conjunto de prueba que no participe en el índice.

In [4]:
from PIL import Image
from backend.ml import retrieve_examples

sample = model['examples'][0]
sample_image = ROOT / 'data' / 'processed' / sample['image']
nearest = retrieve_examples(Image.open(sample_image), limit=1, artifact_path=ROOT / 'models' / 'modelo.joblib')
print('Distancia coseno:', nearest[0]['distance'])
print(nearest[0]['html'][:1200])

Distancia coseno: 2.86102294921875e-06
<html>
<link href="https://cdn.jsdelivr.net/npm/tailwindcss@2.2.19/dist/tailwind.min.css" rel="stylesheet">
<body class="bg-gray-100">
  <div class="flex flex-col items-center justify-center h-screen">
    <h1 class="text-4xl font-bold text-center text-gray-800">Fashion Brand</h1>
    <p class="mt-4 text-lg text-center text-gray-600">
      Fashion Brand is a leading fashion brand that offers a wide range of high-quality, stylish clothing and accessories. Our collections are designed to reflect the latest fashion trends and are made from high-quality materials. We are committed to providing our customers with the best possible shopping experience and we strive to make our collections accessible to all.
    </p>
    <div class="mt-8">
      <img src="https://source.unsplash.com/random/300x200/?fashion" alt="Fashion Brand" class="rounded">
    </div>
    <div class="mt-8">
      <nav class="flex justify-center space-x-4">
        <a href="#" class="

## 5. Iniciar API y frontend

Asegura que MongoDB esté activo y que `.env` tenga una clave de OpenAI con acceso a un modelo de visión. En dos ventanas de Anaconda Prompt, desde la carpeta del proyecto, activa `screenshot-ui` y ejecuta `uvicorn backend.api:app --reload --host 127.0.0.1 --port 8000` en una, y `streamlit run frontend/app.py` en la otra. La aplicación acepta una captura o captura una URL pública, genera los archivos y muestra una vista previa aislada.

In [6]:
import requests

API = 'http://127.0.0.1:8000'
try:
    response = requests.get(API + '/health', timeout=3)
    response.raise_for_status()
    response.json()
except requests.RequestException as exc:
    print('Inicia FastAPI para consultar /health:', type(exc).__name__)

## 6. Evaluación y límites

La prueba de recuperación debe hacerse con capturas separadas de las usadas al entrenar. Para medir la reconstrucción completa, compara estructura, tipografía, color, espaciado y respuesta a distintos anchos. El HTML original no puede deducirse exactamente de una imagen; revisa y valida el código antes de reutilizarlo.